# 09 · FastAPI, CLI contracts, validation, and serialization

Exercise the application boundary without starting a public server or requiring credentials.

In [ ]:
# Local/VS Code and Colab bootstrap. No secret or API key is used.
import json
import os
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    repo = Path("/content/ai-rookie-skill-tree-lab")
    if not repo.exists():
        subprocess.run(
            ["git", "clone", "https://github.com/chelsietao/ai-rookie-skill-tree-lab.git", str(repo)],
            check=True,
        )
    os.chdir(repo)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[full,dev]"], check=True)
elif Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

ROOT = Path.cwd().resolve()
DATA_DIR = ROOT / "data/generated/ci"
METRICS_DIR = ROOT / "artifacts/metrics/ci"
RUN_TRAINING = False  # Set True when you want to regenerate the chapter's metrics.
assert (ROOT / "pyproject.toml").exists(), "Open the repository root or notebooks directory."
print({"environment": "Colab" if IN_COLAB else "local/VS Code", "root": str(ROOT)})

## Boundary design

The package contains reusable domain code. The CLI is a human/automation boundary; FastAPI is an
HTTP boundary. Both delegate instead of duplicating model and data logic.

In [ ]:
import subprocess
cli = Path(sys.executable).parent / "rookie-lab"
help_result = subprocess.run([str(cli), "--help"],
                             capture_output=True, text=True, check=True)
print(help_result.stdout)

## Test the API in-process

`TestClient` runs routing, validation, middleware, and serialization without binding a network port.

In [ ]:
from fastapi.testclient import TestClient
from rookie_lab.serving.api import app
client = TestClient(app)
response = client.get("/health", headers={"X-Request-ID": "notebook-demo"})
print(response.status_code, response.json(), response.headers["X-Request-ID"])

## Input validation and error semantics

In [ ]:
invalid = client.post("/predict/churn", json={"age": 8})
missing = client.get("/metrics/does-not-exist")
unsafe = client.get("/metrics/../secret")
print({"invalid": invalid.status_code, "missing": missing.status_code,
       "unsafe_path": unsafe.status_code})

## Read-only evidence endpoints

In [ ]:
skills = client.get("/skills").json()
print({"official_nodes": skills["active_nodes"], "ai_nodes": skills["ai"]})
print(client.get("/runs").json()["count"])

## JSON and binary serialization

JSON is inspectable and language-neutral. Joblib is convenient for Python pipelines but must never
load an untrusted file because pickle-compatible formats can execute code.

In [ ]:
import tempfile
from rookie_lab.software.serialization import dump_payload, load_payload
payload = {"model": "demo", "metrics": [0.6, 0.7]}
serialization_dir = Path(tempfile.mkdtemp())
for suffix in ("json", "yaml"):
    path = serialization_dir / f"payload.{suffix}"
    dump_payload(payload, path)
    print(suffix, load_payload(path))

## Production gaps

The API binds to localhost by default. It lacks authentication, rate limiting, a feature store,
drift monitoring, rollout policy, and a production model registry. It is a local software contract,
not a claim of production readiness or financial suitability.

In [ ]:
schema = client.get("/openapi.json").json()
rows = [{"path": path, "methods": ", ".join(item)}
        for path, item in schema["paths"].items()]
display(pd.DataFrame(rows).set_index("path"))

## Exercises

Add an API version prefix; return a structured model-unavailable error; write a latency budget test;
design authentication without committing a key; explain why Pydantic validation cannot prevent
distribution shift.